# SVM Component: Personal Loan Prediction

This notebook implements a **Support Vector Machine (SVM)** model for a supervised learning task using the `Bank_Personal_Loan_Modelling.csv` dataset.

## Problem Statement
Predict whether a customer will accept a personal loan (`Personal Loan` = 1) based on customer profile and banking-related features.

## Why SVM?
- SVM is a strong supervised learning algorithm for classification.
- It performs well when classes are separable in a transformed feature space.
- With proper scaling and hyperparameter tuning, it can produce strong decision boundaries.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
data_path = "dataset/Bank_Personal_Loan_Modelling.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nTarget distribution:")
print(df["Personal Loan"].value_counts())

## Preprocessing Decisions

- `Personal Loan` is used as the target variable.
- `ID` is removed because it is only an identifier.
- Negative `Experience` values are clipped to 0 because they are not logically meaningful.
- Numerical features are scaled because SVM is sensitive to feature magnitude.
- Categorical/discrete features are one-hot encoded.
- `class_weight='balanced'` is used because the dataset is imbalanced.

In [ ]:
df = df.copy()
df["Experience"] = df["Experience"].clip(lower=0)

X = df.drop(columns=["Personal Loan", "ID"])
y = df["Personal Loan"]

numeric_features = ["Age", "Experience", "Income", "CCAvg", "Mortgage"]
categorical_features = [
    "ZIP Code",
    "Family",
    "Education",
    "Securities Account",
    "CD Account",
    "Online",
    "CreditCard",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## Model Training and Hyperparameter Tuning

A grid search is used to compare linear and RBF kernels and select the best hyperparameters using the F1-score.

Note: `n_jobs=1` is used for compatibility in some Windows environments.

In [ ]:
svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC(probability=True, class_weight="balanced")),
    ]
)

param_grid = {
    "classifier__kernel": ["linear", "rbf"],
    "classifier__C": [0.1, 1, 10],
    "classifier__gamma": ["scale", "auto"],
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=1,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}

metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
display(metrics_df)

print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - SVM")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve - SVM")
plt.show()

## SVM Decision Boundary Visualization

Because the full dataset has many features, we use **PCA** to project the preprocessed training data into 2 dimensions and then train a 2D SVM only for visualization. This lets you show the SVM contour regions, decision boundary, and margins in a presentation-friendly way.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed)

svm_2d = SVC(kernel="rbf", C=10, gamma="scale")
svm_2d.fit(X_train_2d, y_train)

print("Explained variance by 2 PCA components:", round(pca.explained_variance_ratio_.sum(), 4))

In [ ]:
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
decision_values = svm_2d.decision_function(grid).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, decision_values, levels=30, cmap="coolwarm", alpha=0.35)
plt.contour(xx, yy, decision_values, levels=[-1, 0, 1], colors=["black", "navy", "black"], linestyles=["--", "-", "--"], linewidths=[1.2, 2, 1.2])

scatter = plt.scatter(
    X_train_2d[:, 0],
    X_train_2d[:, 1],
    c=y_train,
    cmap="coolwarm",
    edgecolor="k",
    s=35,
    alpha=0.8,
)

plt.scatter(
    svm_2d.support_vectors_[:, 0],
    svm_2d.support_vectors_[:, 1],
    s=120,
    facecolors="none",
    edgecolors="gold",
    linewidths=1.5,
    label="Support Vectors",
)

plt.title("SVM Decision Boundary, Margins, and Contours (PCA Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(*scatter.legend_elements(), title="Personal Loan")
plt.show()

## Suggested Discussion for the Report

You can explain your SVM component using these points:

1. The task is a binary classification problem where the model predicts whether a customer accepts a personal loan.
2. SVM was selected because it can learn strong separating boundaries after scaling the features.
3. Preprocessing was important because SVM is sensitive to feature scale.
4. The dataset is imbalanced, so F1-score, precision, recall, and ROC-AUC are more informative than accuracy alone.
5. Hyperparameter tuning was used to identify the best kernel and penalty value.
6. A limitation is that SVM is less interpretable than some other models such as decision trees or logistic regression.